In [128]:
!pip install autofe-grass

<h3>Imports</h3>

In [129]:
from sklearn.datasets import fetch_openml
from autofe.stats import GroupAggregationFeatures, StatisticalFeatureGenerator
import pandas as pd
import numpy as np

<h3>Dataset loads</h3>

In [130]:
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame

print(f"Исходный размер данных: {df.shape}")
df.info()

Исходный размер данных: (1309, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   object  
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   object  
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    object  
 10  embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(3), float64(3), int64(3), object(5)
memory usage: 116.8+ KB


In [131]:
df.drop(columns=['survived'], inplace=True)

df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['pclass'] = df['pclass'].astype(int)
df['sibsp'] = df['sibsp'].astype(int)
df['parch'] = df['parch'].astype(int)

numeric_cols = ['age', 'fare', 'sibsp', 'parch']
group_cols = ['pclass']  # Группируем по классу билета

<h3>Basic pipline usage</h3>

In [132]:
group_feats1 = GroupAggregationFeatures(
        numeric_cols=numeric_cols,
        group_cols=group_cols,
        aggs=['mean', 'std'],  # только среднее и стандартное отклонение
        add_deviations=True,   # добавляем diff и ratio
        add_rank=False
)

# sklearn-like fit_transform
df1 = group_feats1.fit_transform(df)

print(f"Исходное количество признаков: {df.shape[1]}")
print(f"Добавлено новых: {df1.shape[1] - df.shape[1]}")

print("\nНовые признаки:")
new_cols = [c for c in df1.columns if c not in df.columns]
for i, col in enumerate(new_cols, 1):
  print(f"  {i:2}. {col}")

Исходное количество признаков: 13
Добавлено новых: 16

Новые признаки:
   1. mean_age
   2. mean_fare
   3. mean_sibsp
   4. mean_parch
   5. std_age
   6. std_fare
   7. std_sibsp
   8. std_parch
   9. diff_mean_age
  10. ratio_mean_age
  11. diff_mean_fare
  12. ratio_mean_fare
  13. diff_mean_sibsp
  14. ratio_mean_sibsp
  15. diff_mean_parch
  16. ratio_mean_parch


<h3>Full pipline usage</h3>



In [133]:
group_feats4 = GroupAggregationFeatures(
        numeric_cols=['age', 'fare'],
        group_cols=['pclass'],
        aggs=['mean', 'std', 'min', 'max'],
        add_deviations=True,
        add_rank=True
)

df_step1 = group_feats4.fit_transform(df)
print(f"Групповые агрегации: {df_step1.shape[1]} признаков")

stat_gen = StatisticalFeatureGenerator(
        numeric_cols=['age', 'fare', 'pclass', 'sibsp', 'parch'],
        unary=['log', 'sqrt'],
        pairwise=['ratio', 'diff'],
        max_features=20,
        corr_th=0.95,
        min_var=1e-5
    )
df_step2 = stat_gen.fit_transform(df_step1)
print(f"Статистические признаки: {df_step2.shape[1]} признаков")

Групповые агрегации: 37 признаков
Статистические признаки: 57 признаков


In [134]:
df_step2.sample(5)

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,...,ratio_pclass_age,ratio_age_sibsp,ratio_sibsp_age,ratio_age_parch,ratio_fare_pclass,ratio_pclass_fare,ratio_fare_sibsp,ratio_sibsp_fare,ratio_fare_parch,ratio_parch_fare
399,2,"Drew, Mr. James Vivian",male,42.0,1,1,28220,32.5000,NaN,S,...,0.047619,42.0,0.023810,42.0,16.250000,0.061538,32.5,0.030769,32.5,0.030769
341,2,"Becker, Miss. Ruth Elizabeth",female,12.0,2,1,230136,39.0000,F4,S,...,0.166667,6.0,0.166667,12.0,19.500000,0.051282,19.5,0.051282,39.0,0.025641
529,2,"Parrish, Mrs. (Lutie Davis)",female,50.0,0,1,230433,26.0000,NaN,S,...,0.040000,NaN,0.000000,50.0,13.000000,0.076923,NaN,0.000000,26.0,0.038462
910,3,"Kallio, Mr. Nikolai Erland",male,17.0,0,0,STON/O 2. 3101274,7.1250,NaN,S,...,0.176471,NaN,0.000000,NaN,2.375000,0.421053,NaN,0.000000,NaN,0.000000
744,3,"Dakic, Mr. Branko",male,19.0,0,0,349228,10.1708,NaN,S,...,0.157895,NaN,0.000000,NaN,3.390267,0.294962,NaN,0.000000,NaN,0.000000
